<a href="https://colab.research.google.com/github/Ariel-hub-121/1142-ML-HW1/blob/main/class_example_decision_tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 實作範例：鐵達尼號生存預測 (分類問題)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# 匯入必要的函式庫
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_train = pd.read_csv('/content/sample_data/train.csv')
df_test  = pd.read_csv('/content/sample_data/test.csv')
df_gender = pd.read_csv('/content/sample_data/gender_submission.csv')
print("--- 資料概覽 (df_train.info()) ---")
df_train.info()
print("--- 資料概覽 (df_test.info()) ---")
df_test.info()
print("--- 資料概覽 (df_gender.info()) ---")
df_gender.info()


# 圖表顯示中文

In [ ]:
# Colab 進行matplotlib繪圖時顯示繁體中文
# 下載台北思源黑體並命名taipei_sans_tc_beta.ttf，移至指定路徑
!wget -O TaipeiSansTCBeta-Regular.ttf https://drive.google.com/uc?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_&export=download

import matplotlib

# 改style要在改font之前
# plt.style.use('seaborn')

matplotlib.font_manager.fontManager.addfont('TaipeiSansTCBeta-Regular.ttf')
matplotlib.rc('font', family='Taipei Sans TC Beta')

# 實作範例：資料探索 - 視覺化分析
* 不同登船口的分佈
* 不同性別的存活人數分析
* 不同艙等的存活率
* 家庭人數與存活率

In [ ]:
#不同登船口的分佈
sns.countplot(x='Embarked', data=df_train)
plt.title('不同登船口的分佈')
plt.show()

#不同性別的存活人數分析
sns.countplot(x='Sex', hue='Survived', data=df_train)
plt.title('不同性別的存活人數分析')
plt.show()

#不同艙等的存活率
sns.countplot(x='Pclass', hue='Survived', data=df_train)
plt.title('不同艙等的存活率')
plt.show()

#家庭人數與存活率
sns.countplot(x='SibSp', hue='Survived', data=df_train)
plt.title('家庭人數與存活率')
plt.show()

print(df_train.columns)

# 實作範例：資料前處理 - 處理缺失值
* 在現實世界的資料中，會遇到缺失值 (NA/NaN) 的情況，需要進行處理，否則模型可能無法正確運行。
1. 年齡 (Age)：
  * 這個欄位有較多缺失值。由於年齡分佈很廣 (從兒童到80歲的長者)，如果使用平均數來填充，容易引入偏差，因此，我們決定使用中位數 (Median) 來取代缺失值，因為中位數對極端值不那麼敏感。
2. 登船港口 (Embarked)：
  * 在資料探索時我們發現，從 'S' 這個登船口登船的人數最多。因此，我們假設缺失的船口最可能是 'S'，並使用眾數 (Mode) 'S' 來填充。
3. 船艙號 (Cabin)：
  * 這個欄位的缺失值非常多，甚至超過了資料的一半。
  * 根據常識，船艙號可能與艙等 (Pclass) 和票價 (Fare) 有關聯，但在我們的分析中，其重要性可能較低。因此，為了簡化模型，我們決定直接刪除 (Drop) 此欄位。


In [ ]:
# 處理缺失值
# 1. 年齡 (Age)：用中位數取代
df_train_copy = df_train.copy() # 複製一份資料
print("before:")
print(df_train_copy.info())
# 檢查缺失值
df_train['Age'] = df_train['Age'].fillna(df_train['Age'].median())
print("after:")
print(df_train.info())


In [ ]:
df_train_copy = df_train.copy() # 複製一份資料
print("before:")
print(df_train_copy.info())
# 2. 登船港口 (Embarked)：用眾數 'S' 取代
# 先找到眾數：df_train['Embarked'].mode() 會回傳Embarked欄位中出現次數最多的值(型態是pandas Series)
# iloc[0]：代表取出第一個眾數
most_frequent_embarked = df_train['Embarked'].mode().iloc[0]
df_train['Embarked'] = df_train['Embarked'].fillna(most_frequent_embarked)
print(df_train.info())

In [ ]:
# 3. 船艙號 (Cabin)：直接刪除
df_train.drop('Cabin', axis=1, inplace=True)



In [ ]:
# 檢查是否還有缺失值
print("處理缺失值後，各欄位的缺失值數量：")
print(df_train.isnull().sum())

# 實作範例：資料前處理 - 特徵工程與編碼
* 建立新特徵 (Feature Engineering)：
  * 有時候原始資料的特徵不足以表達其潛在關係，我們需要創造新的特徵。
  * TravelAlong (是否有家人同行)： 我們觀察到 SibSp (手足/配偶人數) 和 Parch (父母/小孩人數) 這兩個欄位都代表了家庭成員的資訊。在逃生時，家人是兄弟姐妹或父母都一樣重要。因此，我們將這兩個變數合併為一個新的變數 TravelAlong。如果 SibSp 或 Parch 任何一個不為0，則 TravelAlong 為1 (表示有家人同行)，否則為0 (表示獨自一人)。
  * 建立新特徵後，原始的 SibSp 和 Parch 欄位就可以刪除，因為它們的資訊已被 TravelAlong 取代。
* 刪除不必要的特徵：
  * 像 PassengerId (乘客ID)、Name (姓名)、Ticket (船票號碼) 這些欄位屬於個人資料，在做資料分析時通常不重要，因為它們與生存與否的模式關係不大，因此我們會將它們刪除。
* 類別變數編碼 (Encoding)：
  * 機器學習模型通常只能處理數值型資料。因此，我們需要將像「性別 (Sex)」和「登船港口 (Embarked)」這樣的類別變數 (Categorical Variables) 轉換為數值。
  * 我們可以使用 Scikit-learn 中的 LabelEncoder 來完成這件事。它會將不同的類別映射為不同的整數數字。例如，性別可能轉換為0和1 (男/女)，登船港口可能轉換為0, 1, 2。

In [ ]:
from sklearn.preprocessing import LabelEncoder

# 1. 建立新特徵 'TravelAlong'
# 如果 SibSp 或 Parch 任何一個大於 0，則 TravelAlong 為 1，否則為 0
df_train['TravelAlong'] = ((df_train['SibSp'] > 0) | (df_train['Parch'] > 0)).astype(int)

# 2. 刪除不必要的特徵 (包括已被 TravelAlong 取代的 SibSp 和 Parch)
df_train.drop(['PassengerId', 'Name', 'Ticket', 'SibSp', 'Parch'], axis=1, inplace=True)

# 3. 類別變數編碼

# 性別 (Sex)
le_sex = LabelEncoder()
df_train['Sex'] = le_sex.fit_transform(df_train['Sex']) # male/female -> 0/1

# 登船港口 (Embarked)
le_embarked = LabelEncoder()
df_train['Embarked'] = le_embarked.fit_transform(df_train['Embarked']) # C/Q/S -> 0/1/2

print("\n--- 前處理後資料前五筆 (df_train.head()) ---")
print(df_train.head())

print("\n--- 前處理後資料型態 (df_train.info()) ---")
df_train.info()

# 實作範例：模型建立與訓練
1. 區分特徵 (X) 和目標變數 (Y)：
  * X 代表我們用來預測的輸入特徵 (例如：艙等、性別、年齡、票價等)。
  * Y 代表我們想要預測的目標變數 (在本例中是「存活」與否)。
2. 劃分訓練集與測試集：
  * 在建立模型之前，我們會將資料劃分為訓練集 (Training Set) 和測試集 (Testing Set)。
  * 訓練集用於模型學習資料中的模式，而測試集則用於評估模型在未知資料上的表現，模擬真實世界的預測情況。
  * 通常，訓練集佔總資料的70%，測試集佔30%。
3. 載入決策樹分類器：
  * 我們從 sklearn.tree 中匯入 DecisionTreeClassifier。
4. 設定模型參數：
  * max_depth (樹的最大深度) 是決策樹非常重要的參數，它可以限制樹的生長，防止過度學習。
  * 我們不知道樹的最佳深度應該是多深，所以會透過迴圈來設定從2到10的不同深度，看看哪一個深度能讓模型有最好的分類表現。

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 定義特徵 (X) 和目標變數 (Y)
# 移除 'Survived' 欄位作為特徵，保留其他欄位
X = df_train.drop('Survived', axis=1)
# 將 'Survived' 欄位作為目標變數
Y = df_train['Survived']

# 劃分訓練集和測試集
# test_size=0.3 表示測試集佔總資料的30%
# random_state=42 用於確保每次運行結果一致 (可重現性)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

print(f"訓練集特徵 (X_train) 樣本數: {len(X_train)}")
print(f"訓練集目標 (Y_train) 樣本數: {len(Y_train)}")
print(f"測試集特徵 (X_test) 樣本數: {len(X_test)}")
print(f"測試集目標 (Y_test) 樣本數: {len(Y_test)}")

# 實作範例：尋找最佳樹深度
* 為了找到最適合模型的樹深度，我們可以建立一個迴圈，嘗試不同的 max_depth 設定 (從2到10)，並評估每個模型的表現。
* 評估指標：
  * Accuracy (準確率)： 最直觀的指標，表示模型正確預測的比例。
  * Precision (精確率)、Recall (召回率)、F1-Score (F1分數)： 這些是更全面的分類模型評估指標。例如，精確率關注在所有被模型預測為正類別的樣本中，有多少是真正屬於正類別的；召回率則關注所有真正屬於正類別的樣本中，有多少被模型成功找出。

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 嘗試不同深度來尋找最佳模型
best_accuracy = 0
best_depth = 0
model_performance = [] # 用來儲存不同深度模型的表現

# 迴圈從深度 2 到 10 (range(2, 11) 表示從 2 開始，到 10 結束)
for depth in range(2, 11):
    # 建立決策樹分類器模型，設定當前迴圈的 max_depth
    dt_classifier = DecisionTreeClassifier(max_depth=depth, random_state=42)

    # 使用訓練資料來擬合 (Fit) 模型
    dt_classifier.fit(X_train, Y_train)

    # 使用測試資料來進行預測 (Predict)
    Y_pred = dt_classifier.predict(X_test)
    # 獲取預測機率，roc_auc_score 需要機率值
    Y_pred_proba = dt_classifier.predict_proba(X_test)

    # 計算模型的評估指標
    accuracy = accuracy_score(Y_test, Y_pred)
    precision = precision_score(Y_test, Y_pred, zero_division=0)
    recall = recall_score(Y_test, Y_pred, zero_division=0)
    f1 = f1_score(Y_test, Y_pred, zero_division=0)
    # 計算 AUC
    auc_score = roc_auc_score(Y_test, Y_pred_proba[:, 1]) # Y_pred_proba[:, 1] 是預測為正類 (Survived=1) 的機率

    # 將結果儲存到列表中
    model_performance.append({
        'Depth': depth,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'AUC': auc_score
    })

    # 如果當前模型的準確率比之前的最佳準確率更高，則更新最佳深度和準確率
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_depth = depth

print("\n--- 不同樹深度下的模型表現 ---")
for perf in model_performance:
    # 格式化輸出，保留四位小數
    print(f"深度: {perf['Depth']}, 準確率: {perf['Accuracy']:.4f}, "
          f"精確率: {perf['Precision']:.4f}, 召回率: {perf['Recall']:.4f}, "
          f"F1分數: {perf['F1-Score']:.4f}, AUC: {perf['AUC']:.4f}")

print(f"\n根據準確率，最佳樹深度為: {best_depth} (準確率: {best_accuracy:.4f})")
# 根據來源的範例，最佳深度為6，準確率為0.813433

# 實作範例：模型視覺化與預測
* 訓練好的決策樹模型可以被視覺化，這對於理解模型的決策邏輯非常有幫助。
* `export_graphviz`： Scikit-learn 提供這個函數，可以將決策樹輸出為 `.dot` 格式的檔案。這個 `.dot` 檔案可以被 Graphviz 軟體讀取並轉換成圖片檔 (例如：`.png` 或 `.jpg`)。
* 如何解讀決策樹圖：
  * 圖中的每個方框代表一個節點。
  * 每個節點內會顯示用於切分的特徵條件 (例如：`Sex <= 0.5`)。
  * 根據條件判斷結果，資料會流向左邊 (通常是 True) 或右邊 (通常是 `False`) 的子節點。
  * 最終到達的葉節點會顯示該路徑下的最終分類結果 (例如：`class = Survived`)。

## 安裝套件（Graphviz + pydotplus）


In [ ]:
# 安裝套件（Graphviz + pydotplus）
!apt-get install -y graphviz
!pip install graphviz pydotplus

## 模型視覺化

In [ ]:
# 匯入套件
from sklearn.tree import export_graphviz
import pydotplus
from IPython.display import Image
import graphviz

# 重新訓練最佳模型（保險起見）
final_dt_classifier = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
final_dt_classifier.fit(X_train, Y_train)

# 特徵與類別名稱
feature_names = X.columns.tolist()
class_names = ['Not Survived', 'Survived']

# ➤ 1. 儲存為 .dot 檔案（可供下載或外部轉檔用）
export_graphviz(
    final_dt_classifier,
    out_file='titanic_decision_tree.dot',
    feature_names=feature_names,
    class_names=class_names,
    filled=True,
    rounded=True,
    special_characters=True
)

print(".dot 檔案已儲存為 'titanic_decision_tree.dot'")

# ➤ 2. 轉成圖形，inline 顯示在 Colab/Jupyter 中
dot_data = export_graphviz(
    final_dt_classifier,
    out_file=None,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,
    rounded=True,
    special_characters=True
)

graph = pydotplus.graph_from_dot_data(dot_data)
Image(graph.create_png())

### 將 .dot 檔案轉換成 .png 指令

In [ ]:
!dot -Tpng titanic_decision_tree.dot -o titanic_decision_tree.png

# 實作範例 : 模型預測
* 假設我們要預測一個新的乘客是否存活，這裡我們需要輸入所有特徵的值，並且確保順序和前處理方式一致
* 假設新乘客資料: Pclass=3, Sex=male(1), Age=30, Fare=15, Embarked=S(2), TravelAlong=0(No)
* **注意：這裡的數值必須是經過 LabelEncoder 編碼後的結果**

In [ ]:
new_passenger_features = pd.DataFrame([
    [3, 1, 30, 15.0, 2, 0]],
    columns=['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'TravelAlong'])

# 使用訓練好的模型進行預測
predicted_survival = final_dt_classifier.predict(new_passenger_features)
# 使用 predict_proba 估計屬於各類別的機率
predicted_proba = final_dt_classifier.predict_proba(new_passenger_features)

print(f"\n預測新乘客的存活結果: {predicted_survival} (0: 死亡, 1: 存活)")
print(f"預測為死亡的機率: {predicted_proba[0][0]:.4f}")
print(f"預測為存活的機率: {predicted_proba[0][1]:.4f}")

# 實作範例：迴歸問題 - 波士頓房價預測
* 決策樹不僅可以用於分類問題，也可以用於迴歸問題，這時我們使用決策樹迴歸器 (DecisionTreeRegressor)。
* 模型： 從 sklearn.tree 匯入 DecisionTreeRegressor。
* 評估指標：
  * 迴歸模型的績效通常以最小極值為主。
  * 常見的迴歸評估指標包括：
    * R² (決定係數)： 衡量模型解釋變異的比例，值越接近1越好。
    * MAE (平均絕對誤差)： 預測值與真實值之間絕對差值的平均值，越小越好。
    * MSE (均方誤差)： 預測值與真實值之間平方差值的平均值，對大誤差懲罰較重，越小越好。
    * RMSE (均方根誤差)： MSE 的平方根，與原始數據單位相同，越小越好。

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_openml
import pandas as pd
import numpy as np

# 載入波士頓房價資料集
boston = fetch_openml(name='boston', version=1, as_frame=True)
X_boston = pd.DataFrame(boston.data, columns=boston.feature_names)
Y_boston = pd.Series(boston.target, name='MEDV')
print("已成功載入波士頓房價資料集。")

In [ ]:
X_boston.head(5)

In [ ]:
Y_boston.head(5)

In [ ]:
# 劃分訓練集和測試集
X_train_boston, X_test_boston, Y_train_boston, Y_test_boston = \
    train_test_split(X_boston, Y_boston, test_size=0.3, random_state=42)


print(f"訓練集特徵 (X_train_boston) 樣本數: {len(X_train_boston)}")
print(f"訓練集目標 (Y_train_boston) 樣本數: {len(Y_train_boston)}")
print(f"測試集特徵 (X_test_boston) 樣本數: {len(X_test_boston)}")
print(f"測試集目標 (Y_test_boston) 樣本數: {len(Y_test_boston)}")

In [ ]:
# 建立決策樹迴歸模型
# max_depth 參數同樣重要，可以防止過度學習
dt_regressor = DecisionTreeRegressor(max_depth=5, random_state=42)
dt_regressor.fit(X_train_boston, Y_train_boston)

# 進行預測
Y_pred_boston = dt_regressor.predict(X_test_boston)

# 評估模型表現
r2 = r2_score(Y_test_boston, Y_pred_boston)
mae = mean_absolute_error(Y_test_boston, Y_pred_boston)
mse = mean_squared_error(Y_test_boston, Y_pred_boston)
rmse = np.sqrt(mse) # RMSE 是 MSE 的平方根

print("\n--- 決策樹迴歸模型表現 ---")
print(f"R² (決定係數): {r2:.4f}")
print(f"MAE (平均絕對誤差): {mae:.4f}")
print(f"MSE (均方誤差): {mse:.4f}")
print(f"RMSE (均方根誤差): {rmse:.4f}")
